# FEMA Raw Data Quality

**Purpose.** Audit the files ingested for this provider and the corresponding `raw.*`
DuckDB tables before any normalization, blending, or analytical transformation.

This notebook covers the supplied files/tables, observation grain, date and geography
coverage, column types and meanings, missingness and suppression, duplicate/invalid
keys, numeric ranges, suspicious values, source limitations, and downstream readiness.

## Setup and provider rules

In [1]:
from pathlib import Path
import re
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

ROOT = Path.cwd()
while not (ROOT / "data" / "quoll.duckdb").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DB_PATH = ROOT / "data" / "quoll.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)

PROVIDER = 'fema'
TABLE_PATTERNS = ['fema_%', 'nri_table_counties']
PRIMARY_PATTERNS = ['fema_disaster_declarations', 'fema_web_disaster_summaries', 'nri_table_counties']
KEY_CANDIDATES = [['disasterNumber', 'fipsStateCode', 'fipsCountyCode', 'declarationDate'], ['STCOFIPS'], ['disasterNumber']]
DATE_CANDIDATES = ['declarationDate', 'incidentBeginDate', 'incidentEndDate', 'lastRefresh']
GEO_CANDIDATES = ['state', 'fipsStateCode', 'fipsCountyCode', 'STCOFIPS']
NUMERIC_HINTS = ['disasterNumber', 'RISK_SCORE', 'EAL_SCORE', 'SOVI_SCORE', 'RESL_SCORE', 'totalAmountIhpApproved']
SUPPRESSION_CODES = ['', 'N/A', 'null']

def matches(name, patterns):
    return any(re.fullmatch(pattern.replace("%", ".*"), name, flags=re.I) for pattern in patterns)

def qi(value):
    return '"' + value.replace('"', '""') + '"'

raw_tables = con.execute(
    "SELECT table_name FROM information_schema.tables "
    "WHERE table_schema = 'raw' ORDER BY table_name"
).df()["table_name"].tolist()
provider_tables = [name for name in raw_tables if matches(name, TABLE_PATTERNS)]
primary_tables = [name for name in provider_tables if matches(name, PRIMARY_PATTERNS)]
provider_tables, primary_tables

(['fema_disaster_declarations',
  'fema_web_disaster_summaries',
  'nri_table_counties'],
 ['fema_disaster_declarations',
  'fema_web_disaster_summaries',
  'nri_table_counties'])

## Files and tables supplied

In [2]:
file_inventory = con.execute(
    '''
    SELECT table_name, filename, source_folder, source_path,
           loaded_at, row_count, detected_columns,
           upstream_source_url, content_sha256
    FROM meta.files
    WHERE table_schema = 'raw'
    ORDER BY table_name
    '''
).df()
file_inventory = file_inventory.loc[file_inventory["table_name"].isin(provider_tables)]

table_rows = []
for table in provider_tables:
    row_count = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    column_count = con.execute(
        "SELECT count(*) FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).fetchone()[0]
    table_rows.append({"table_name": table, "rows": row_count, "columns": column_count,
                       "primary_data_table": table in primary_tables})
table_inventory = pd.DataFrame(table_rows)
display(file_inventory)
display(table_inventory)

,table_name,filename,source_folder,source_path,loaded_at,row_count,detected_columns,upstream_source_url,content_sha256
48,fema_disaster_declarations,FEMA_Disaster_Declarations.csv,fema,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:55:39.710742+00:00,69546,"[""column00"", ""id"", ""disasterNumber"", ""state"", ...",https://www.fema.gov/api/open/v2/DisasterDecla...,08a8f0ddce01386721d7fd0355a1071a3748afe383ac07...
49,fema_web_disaster_summaries,FemaWebDisasterSummaries.csv,climate_damage\raw\fema_web_disaster_summaries,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:55:39.891231+00:00,3955,"[""disasterNumber"", ""totalNumberIaApproved"", ""t...",https://www.fema.gov/api/open/v1/FemaWebDisast...,3e053c7d843beab4033e47aa67dee8c220d843a129ac59...
54,nri_table_counties,NRI_Table_Counties.csv,fema,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:55:38.703512+00:00,3232,"[""OID_"", ""NRI_ID"", ""STATE"", ""STATEABBRV"", ""STA...",https://hazards.fema.gov/nri/data-resources,f6640493f6225f2bee7484c698878807299a0081a1aa16...


,table_name,rows,columns,primary_data_table
0,fema_disaster_declarations,69546,29,True
1,fema_web_disaster_summaries,3955,14,True
2,nri_table_counties,3232,465,True


## Observation grain

Declarations are declaration-area records; web summaries are disaster-level financial summaries; NRI is one county snapshot per NRI release.

The checks below infer candidate keys from the raw columns. A repeated candidate key is
reported rather than silently removed because some provider tables legitimately contain
additional dimensions.

## Column types and meanings

In [3]:
schema_frames = []
for table in primary_tables:
    schema = con.execute(f"DESCRIBE raw.{qi(table)}").df()
    schema.insert(0, "table_name", table)
    schema["inferred_meaning"] = (
        schema["column_name"].str.replace("_", " ", regex=False)
        .str.replace(r"(?<=[a-z])(?=[A-Z])", " ", regex=True)
        .str.strip()
    )
    schema_frames.append(schema)
schema_inventory = pd.concat(schema_frames, ignore_index=True) if schema_frames else pd.DataFrame()
display(schema_inventory)

,table_name,column_name,column_type,null,key,default,extra,inferred_meaning
0,fema_disaster_declarations,column00,VARCHAR,YES,None,None,None,column00
1,fema_disaster_declarations,id,VARCHAR,YES,None,None,None,id
2,fema_disaster_declarations,disasterNumber,VARCHAR,YES,None,None,None,disaster Number
3,fema_disaster_declarations,state,VARCHAR,YES,None,None,None,state
4,fema_disaster_declarations,femaDeclarationString,VARCHAR,YES,None,None,None,fema Declaration String
...,...,...,...,...,...,...,...,...
503,nri_table_counties,WNTW_ALR_NPCTL,VARCHAR,YES,None,None,None,WNTW ALR NPCTL
504,nri_table_counties,WNTW_RISKV,VARCHAR,YES,None,None,None,WNTW RISKV
505,nri_table_counties,WNTW_RISKS,VARCHAR,YES,None,None,None,WNTW RISKS
506,nri_table_counties,WNTW_RISKR,VARCHAR,YES,None,None,None,WNTW RISKR


## Date and geographic coverage

In [4]:
coverage_rows = []
for table in primary_tables:
    columns = con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"].tolist()
    row = {"table_name": table}
    for column in DATE_CANDIDATES:
        if column in columns:
            normalized_column = column.lower()
            if normalized_column == "year" or normalized_column.endswith("_year"):
                coverage_type = "INTEGER"
            elif normalized_column == "month" or normalized_column.endswith("_month"):
                coverage_type = "INTEGER"
            else:
                coverage_type = "TIMESTAMP"
            values = con.execute(
                f"SELECT min(try_cast({qi(column)} AS {coverage_type})), "
                f"max(try_cast({qi(column)} AS {coverage_type})) "
                f"FROM raw.{qi(table)}"
            ).fetchone()
            row[f"{column}_min"] = values[0]
            row[f"{column}_max"] = values[1]
    for column in GEO_CANDIDATES:
        if column in columns:
            row[f"{column}_distinct"] = con.execute(
                f"SELECT count(DISTINCT {qi(column)}) FROM raw.{qi(table)}"
            ).fetchone()[0]
    coverage_rows.append(row)
coverage = pd.DataFrame(coverage_rows)
display(coverage)

,table_name,declarationDate_min,declarationDate_max,incidentBeginDate_min,incidentBeginDate_max,incidentEndDate_min,incidentEndDate_max,lastRefresh_min,lastRefresh_max,state_distinct,fipsStateCode_distinct,fipsCountyCode_distinct,STCOFIPS_distinct
0,fema_disaster_declarations,1953-05-02,2026-01-24,1953-05-02,2026-01-23,1953-05-02,2025-12-30,2024-08-27 18:22:14.800,2026-01-30 14:01:19.257,59.0,59.0,347.0,NaN
1,fema_web_disaster_summaries,NaT,NaT,NaT,NaT,NaT,NaT,2023-03-18 13:22:12.883,2026-06-25 03:02:13.686,NaN,NaN,NaN,NaN
2,nri_table_counties,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaN,NaN,NaN,3232.0


## Missingness and suppression codes

In [5]:
missing_rows = []
suppression_rows = []
suppression_sql = ", ".join("?" for _ in SUPPRESSION_CODES)
for table in primary_tables:
    columns = con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=? ORDER BY ordinal_position", [table]
    ).df()["column_name"].tolist()
    row_count = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    # Profile all columns for compact tables and the first 80 for unusually wide sources.
    for column in columns[:80]:
        null_count, blank_count = con.execute(
            f"SELECT count(*) FILTER (WHERE {qi(column)} IS NULL), "
            f"count(*) FILTER (WHERE trim(cast({qi(column)} AS VARCHAR))='') "
            f"FROM raw.{qi(table)}"
        ).fetchone()
        missing_rows.append({
            "table_name": table, "column_name": column,
            "missing_count": null_count + blank_count,
            "missing_pct": (null_count + blank_count) / row_count * 100 if row_count else np.nan,
        })
        if SUPPRESSION_CODES:
            suppressed = con.execute(
                f"SELECT count(*) FROM raw.{qi(table)} "
                f"WHERE trim(cast({qi(column)} AS VARCHAR)) IN ({suppression_sql})",
                SUPPRESSION_CODES,
            ).fetchone()[0]
            if suppressed:
                suppression_rows.append({
                    "table_name": table, "column_name": column,
                    "suppression_or_sentinel_count": suppressed,
                })
missingness = pd.DataFrame(missing_rows).sort_values(
    ["missing_pct", "table_name"], ascending=[False, True]
)
suppression = pd.DataFrame(suppression_rows)
display(missingness)
display(suppression if not suppression.empty else pd.DataFrame(
    {"result": ["No configured literal suppression codes were present in profiled columns; nulls remain material."]}
))

,table_name,column_name,missing_count,missing_pct
103,nri_table_counties,CFLD_EVNTS,3232,100.000000
32,fema_web_disaster_summaries,totalAmountHaApproved,3405,86.093552
33,fema_web_disaster_summaries,totalAmountOnaApproved,3351,84.728192
30,fema_web_disaster_summaries,totalNumberIaApproved,3349,84.677623
31,fema_web_disaster_summaries,totalAmountIhpApproved,3349,84.677623
...,...,...,...,...
90,nri_table_counties,AVLN_HLRR,0,0.000000
96,nri_table_counties,AVLN_EALR,0,0.000000
102,nri_table_counties,AVLN_RISKR,0,0.000000
112,nri_table_counties,CFLD_HLRR,0,0.000000


,result
0,No configured literal suppression codes were p...


## Duplicate or invalid keys

In [6]:
key_rows = []
for table in primary_tables:
    columns = set(con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"])
    keys = next((candidate for candidate in KEY_CANDIDATES if set(candidate).issubset(columns)), [])
    if not keys:
        key_rows.append({"table_name": table, "candidate_key": None,
                         "duplicate_key_groups": np.nan, "invalid_key_rows": np.nan})
        continue
    key_expr = ", ".join(qi(column) for column in keys)
    invalid = " OR ".join(
        f"{qi(column)} IS NULL OR trim(cast({qi(column)} AS VARCHAR))=''" for column in keys
    )
    duplicate_groups = con.execute(
        f"SELECT count(*) FROM (SELECT {key_expr}, count(*) n "
        f"FROM raw.{qi(table)} GROUP BY {key_expr} HAVING count(*) > 1)"
    ).fetchone()[0]
    invalid_rows = con.execute(
        f"SELECT count(*) FROM raw.{qi(table)} WHERE {invalid}"
    ).fetchone()[0]
    key_rows.append({"table_name": table, "candidate_key": " + ".join(keys),
                     "duplicate_key_groups": duplicate_groups,
                     "invalid_key_rows": invalid_rows})
key_quality = pd.DataFrame(key_rows)
display(key_quality)

,table_name,candidate_key,duplicate_key_groups,invalid_key_rows
0,fema_disaster_declarations,disasterNumber + fipsStateCode + fipsCountyCod...,254,0
1,fema_web_disaster_summaries,disasterNumber,0,0
2,nri_table_counties,STCOFIPS,0,0


## Numeric ranges and suspicious values

In [7]:
numeric_rows = []
for table in primary_tables:
    columns = set(con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"])
    for column in [name for name in NUMERIC_HINTS if name in columns]:
        numeric = (
            f"try_cast(replace(trim(cast({qi(column)} AS VARCHAR)), ',', '') AS DOUBLE)"
        )
        result = con.execute(
            f"SELECT count(*) FILTER (WHERE {numeric} IS NOT NULL), "
            f"min({numeric}), max({numeric}), "
            f"count(*) FILTER (WHERE {numeric} < 0) "
            f"FROM raw.{qi(table)}"
        ).fetchone()
        numeric_rows.append({
            "table_name": table, "column_name": column,
            "numeric_count": result[0], "minimum": result[1],
            "maximum": result[2], "negative_count": result[3],
            "review_flag": (
                "review negative values/sentinels" if result[3] else
                "review extreme min/max against provider definition"
            ),
        })
numeric_ranges = pd.DataFrame(numeric_rows)
display(numeric_ranges)

,table_name,column_name,numeric_count,minimum,maximum,negative_count,review_flag
0,fema_disaster_declarations,disasterNumber,69546,1.000000,5.615000e+03,0,review extreme min/max against provider defini...
1,fema_web_disaster_summaries,disasterNumber,3955,1239.000000,5.640000e+03,0,review extreme min/max against provider defini...
2,fema_web_disaster_summaries,totalAmountIhpApproved,606,1756.200000,5.247169e+09,0,review extreme min/max against provider defini...
3,nri_table_counties,RISK_SCORE,3144,0.031807,1.000000e+02,0,review extreme min/max against provider defini...
4,nri_table_counties,EAL_SCORE,3232,0.030941,1.000000e+02,0,review extreme min/max against provider defini...
5,nri_table_counties,SOVI_SCORE,3144,10.623410,1.000000e+02,0,review extreme min/max against provider defini...
6,nri_table_counties,RESL_SCORE,3144,0.031807,1.000000e+02,0,review extreme min/max against provider defini...


## Source-specific limitations

FEMA declarations are administrative records rather than direct measures of hazard intensity. A county incident can appear under more than one declaration, missing incident end dates require an explicit rule, and assistance totals reflect program administration and eligibility.

## Downstream readiness

**Assessment: PASS WITH LIMITATIONS. Declaration records require county-event deduplication and incident-type exclusions before event-window use; NRI ratings are treated as a current county risk grouping.**

This assessment is conditional on the displayed inventories and checks. The normalized
`mart.*` builders—not this notebook—own parsing, suppression handling, geographic
resolution, deduplication, and downstream transformations.

In [8]:
con.close()